# Binary Neutron Star Tidal Model
Written by: Sophia Owen

This notebook is used to save all three animations externally, as opposed to displaying them within the notebook.

Contains code for the following animations:

1. Stationary reference frame with point mass companion
2. Corotating reference frame with point mass companion
3. Corotating reference frame with tidally perturbed companion

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings("ignore")   # it will tell you sp.special.sph_harm is deprecated every single frame of the animation without this line

In [3]:
# constants (with units)
M = 1.35*1.989*10**30   # kg
M_prime = M   # kg
R = 12000   # m
Ia = 0.75   # units?
mu = M*M_prime/(M+M_prime)   # =1/2 while M_prime = M
G = 6.67*10**(-11)   # m^3 s^-2 kg^-1
wa2 =0.8*M*G/(R**3)   # (rad/s)^2
c = 299792458   # m/s

# natural unit definitions
#------free units------
M_u = 1.989*10**30   # mass scale, kg, 1 solar mass
R_u = 10000   # length scale, m
G_u = G   # scale G/G_u=1
#-----dependent units-----
wa_u = np.sqrt(G*M_u/(R_u**3))   # rad/s
T_u = 1/wa_u   # s
V_u = R_u*wa_u   # m/s

# naturalized constants (unitless)
M_nat = M/M_u
M_p_nat = M_prime/M_u
R_nat = R/R_u
wa2_nat = wa2/(wa_u**2)
G_nat = G/G_u
c_nat = c/V_u

## ODEs
We solve a system of second order ordinary differential equations whose solution curves show the evolution of parameters of the BNS system. The tidal lag angle $\phi$ and the orbital separation distance $r$ are given as follows:

$$\ddot \phi = \frac{-2 \dot r \dot \phi}{r}-\frac{32}{5} \frac{G \mu r^2}{c^5} \dot \phi^5$$

$$\ddot r = r \dot \phi^2 - \frac{G(M+M')}{r^2}$$

where $M$ is the mass of the deformed NS, $M'$ is the mass of the companion, and $\mu$ is the reduced mass given by $\mu = \frac{MM'}{M+M'}$. The tidal perturbation $q_a$ dependent on $\phi$ and $r$:

$$\ddot q_a = \omega_a^2 I_a \frac{M'}{M} (\frac{R}{r})^3e^{-mi\phi} - \omega_a^2 q_a$$

where $I_a$ is the tidal overlap integral, $\omega_a$ is the angular velocity of the deformed star, and both stars have radius $R$. We stop integration at $r=2R$, when the surfaces of the stars make contact. Converting the system of three second order ODEs into a system of six first order ODEs gives the following system:

$$\dot r = r_1$$
$$\dot r_1 = r\phi_1^2 - \frac{G(M+M')}{r^2}$$
$$\dot \phi = \phi_1$$
$$\dot \phi_1 = \frac{-2 r_1 \phi_1}{r}-\frac{32}{5} \frac{G \mu r^2}{c^5} \phi_1^5$$
$$\dot q_a = q_1$$
$$\dot q_1 = \omega_a^2 I_a \frac{M'}{M} (\frac{R}{r})^3e^{-mi\phi} - \omega_a^2 q_a$$

Because $q_a$ is dependent on $m = -2, 0, 2$, this code uses a system of 10 ODEs. Numerical integration takes place using natural units.  

In [4]:
#---------------Solving System of ODEs---------------+

# naturalized initial conditions
r0 = 4*R_nat
r10 = 0
phi0 = 0
phi10 = np.sqrt(G_nat*(M_nat+M_p_nat)/(r0**3))
qa0_2 = (wa2_nat/(wa2_nat - 4*phi10**2))*Ia*(M_p_nat/M_nat)*((R_nat/r0)**3)*np.exp(-2j*phi0)
q10_2 = -2j*phi10*qa0_2
qa0_0 = Ia*(M_p_nat/M_nat)*((R_nat/r0)**3)   # (wa2_nat/(wa2_nat - phi10**2))* = 1 and *np.exp(-2j*phi0) = 1
q10_0 = 0   # -2j*phi10*qa0 = 0
qa0_n2 = (wa2_nat/(wa2_nat - 4*phi10**2))*Ia*(M_p_nat/M_nat)*((R_nat/r0)**3)*np.exp(2j*phi0)
q10_n2 = 2j*phi10*qa0_n2

init_vals = np.array([r0, r10, phi0, phi10, qa0_2, q10_2,  qa0_0, q10_0,  qa0_n2, q10_n2], dtype=np.complex128)

# ODE solver function
def odes(t, y):   
    '''
    description:
    - solves combined system of ODEs for m = -2, 0, 2
    - produces 10 solution curves
    --------------
    notes:
    - in variable names, n2 indicates m = -2
    - input into odes(t, y) in natural units, converted to physical units for calc, converted back to natural units for output
    '''
    
    r, r1, phi, phi1, qa_2, q1_2, qa_0, q1_0, qa_n2, q1_n2 = y   # natural units

    # convert to physical units for calculations
    r_phys = r*R_u   
    r1_phys = R_u*r1/T_u
    phi_phys = phi   # radians unitless
    phi1_phys = phi1/T_u
    qa_phys_2 = qa_2   # radians unitless
    q1_phys_2 = q1_2/T_u
    qa_phys_0 = qa_0 
    q1_phys_0 = q1_0/T_u
    qa_phys_n2 = qa_n2 
    q1_phys_n2 = q1_n2/T_u

    # calculations in physical units
    drdt = r1_phys   # r1 = r', r1' = r''
    dr1dt = (r_phys*phi1_phys**2)-G*(M+M_prime)/(r_phys**2) 
    dphi_dt = phi1_phys   # phi1 = phi', phi1' = phi''
    dphi1_dt = (-2*r1_phys*phi1_phys/r_phys)-(32/5)*(G*mu*(r_phys**2)*(phi1_phys**5)/(c**5))
    dqadt_2 = q1_phys_2   # q1 = qa', q1' = qa''
    dq1dt_2 = wa2*Ia*(M_prime/M)*((R/r_phys)**3)*(np.exp(-2j*phi_phys))-wa2*qa_phys_2
    dqadt_0 = q1_phys_0   # q1 = qa', q1' = qa''
    dq1dt_0 = wa2*Ia*(M_prime/M)*((R/r_phys)**3)-wa2*qa_phys_0   #*(np.exp(-2j*phi_phys)) = 1
    dqadt_n2 = q1_phys_n2   # q1 = qa', q1' = qa''
    dq1dt_n2 = wa2*Ia*(M_prime/M)*((R/r_phys)**3)*(np.exp(2j*phi_phys))-wa2*qa_phys_n2

    # return naturalized values
    return [(T_u/R_u)*drdt, (T_u**2/R_u)*dr1dt, T_u*dphi_dt, (T_u**2)*dphi1_dt, T_u*dqadt_2, (T_u**2)*dq1dt_2, T_u*dqadt_0, (T_u**2)*dq1dt_0, T_u*dqadt_n2, (T_u**2)*dq1dt_n2]

# event definition (stop integration)
def stopIntegration(t, y): 
    return y[0]-2*R_nat

stopIntegration.terminal = True   # explicitly stop integration instead of noting the event and continuing integration
stopIntegration.direction = 0   # stop when r(t)-2R=0

t_vals = (0, 1700)   # ~1327 time values once stopIntegration is reached

# solve the system, call sol.t or sol.y[] for solution curves
sol = solve_ivp(odes, t_vals, init_vals, events=stopIntegration, method='RK45', rtol=1e-8, atol=1e-10) 

In [5]:
# get all 5 solution curves for animation
dist = sol.y[0]
sep_angle = sol.y[2]
qa_complex_2 = sol.y[4]
qa_complex_0 = sol.y[6]
qa_complex_n2 = sol.y[8]
qa_real_2 = sol.y[4].real

## Stationary Reference Frame Animation
We model quadrupolar tidal deformation in one NS and treat the companion as a point mass. In this animation, the observer moves with the deformed star in its orbit, but does not rotate with the star.

### Coordinates of the Deformed NS
Using linear perturbation theory, we define the vector $\vec{\xi}$ to be the displacement of any infinitesimally small unit of mass from its non-deformed position within the tidally deformed star:

$$\vec{\xi} = Re(R \sum\limits_{m=-2,0,2} q_a W_{l,m} Y_{l,m}) \hat{r} + Im(\frac{R^2}{l} \sum\limits_{m=-2,0,2} q_a W_{l,m} \nabla Y_{l,m})$$ 

where $q_a$ comes from the ODE solutions above. We define $W_{l,m}$ and $Y_{l,m}$ respectively:

$$W_{l,m} = \frac{4\pi}{2l+1}Y_{l,m}$$

$$Y_{l,m} = (-1)^m \sqrt{\frac{(2l+1)(l+m)!}{4\pi (l-m)!}}e^{im\varphi} P_{l,m}(\cos{\theta})$$

where $W_{l,m}$ is a normalization constant at $\varphi = 0$, $Y_{l,m}$ is the spherical harmonic equation, and $P_{l,m}(\cos{\theta})$ are the Legendre polynomials. We fix $l=2$ and $\theta = \frac{\pi}{2}$, and vary $\varphi \in [0,2 \pi ]$. We then define the coordinates of the deformed star in cartesian coordinates:

$$x=\xi_r \cos{\varphi}-\xi_h \sin{\varphi}+R\cos{\varphi}$$

$$y=\xi_r \sin{\varphi}+\xi_h \cos{\varphi}+R\sin{\varphi}$$

### Coordinates of the Companion Star
We define the coordinates of the companion star using the solutions to the ODEs above:

$$x_{comp} = r\cos{\phi}$$

$$y_{comp} = r\sin{\phi}$$

Note that $\varphi$ and $\phi$ are not the same variable.

In [6]:
# variables for tidally deformed star
l = 2
m = [-2, 0, 2]
theta = np.pi/2
phi = np.linspace(0, 2*np.pi, 200)
R = 1   # radius of both stars  

# monopole (plotted in red for reference)
monopole = np.sqrt(4*np.pi)*sp.special.sph_harm(0, 0, phi, theta)   # (4pi/(2l+1))*ylm where l=0 and m=0 is sqrt(4pi)
x1 = monopole*np.cos(phi)
y1 = monopole*np.sin(phi)

In [7]:
#---------------Calculation Functions---------------+

def ylm(m):
    '''
    description:
    function to calculate spherical harmonics with varying input m
    '''
    return sp.special.sph_harm(m, l, phi, theta)

def wlm(m):
    '''
    description:
    function to calculate normalization factors with varying input m
    '''
    return (4*np.pi/5)*sp.special.sph_harm(m, l, 0, theta)   # 5 = 2l + 1

def findMajorAxis(xArray, yArray):
    '''
    description:
    function to find the major axis of each perturbed star
    --------------
    input:
    xArray (ndarray): x coordinates of perturbed star
    yArray (ndarray): y coordinates of perturbed star
    '''
    greatestDist = 0
    xCoordinate = 0
    yCoordinate = 0
    for i in range (len(xArray)):
        dist = np.sqrt(xArray[i]**2 + yArray[i]**2)
        if dist > greatestDist:
            greatestDist = dist
            xCoordinate = xArray[i]
            yCoordinate = yArray[i]
    coordinates = [xCoordinate, yCoordinate]
    return coordinates

# 
def findDynamicRad_V6(qa_2, qa_0, qa_n2):
    '''
    description:
    - function to calculate xi coordinates for each animation frame
    - use with plotV6, plotV8, plotV11, and plotV12 depending on the animation 
    --------------
    input:
    qa_2 (float): value for time t from the ode solution curve qa for m=2 (t specified in update function)
    qa_0 (float): value for time t from the ode solution curve qa for m=0
    qa_n2 (float): value for time t from the ode solution curve qa for m=-2
    '''
    xi_r = R*(wlm(m[0])*qa_n2*ylm(m[0])+wlm(m[1])*qa_0*ylm(m[1])+wlm(m[2])*qa_2*ylm(m[2])) 
    xi_phi = (R**2/l)*(np.sqrt(15/(32*np.pi))*(wlm(m[0])*qa_n2*2j*np.exp(2j*phi)-wlm(m[2])*qa_2*2j*np.exp(-2j*phi)))
    xi_x = xi_r.real*np.cos(phi) - xi_phi.imag*np.sin(phi) + R*np.cos(phi)
    xi_y = xi_r.real*np.sin(phi) + xi_phi.imag*np.cos(phi) + R*np.sin(phi)
    xi = [xi_x, xi_y]
    return xi

def findDynamicQuad(qa_2, qa_0, qa_n2): 
    '''
    description:
    - function to calculate xi with only quadrupole contribution
    - use with plotV10
    --------------
    input:
    qa_2 (float): value for time t from the ode solution curve qa for m=2 (t specified in update function)
    qa_0 (float): value for time t from the ode solution curve qa for m=0
    qa_n2 (float): value for time t from the ode solution curve qa for m=-2
    '''
    xi_r = R*(wlm(m[0])*qa_n2*ylm(m[0])+wlm(m[1])*qa_0*ylm(m[1])+wlm(m[2])*qa_2*ylm(m[2])) 
    xi_phi = (R**2/l)*(np.sqrt(15/(32*np.pi))*(wlm(m[0])*qa_n2*2j*np.exp(2j*phi)-wlm(m[2])*qa_2*2j*np.exp(-2j*phi)))
    xi_x = xi_r.real*np.cos(phi) - xi_phi.imag*np.sin(phi) 
    xi_y = xi_r.real*np.sin(phi) + xi_phi.imag*np.cos(phi)
    xi = [xi_x, xi_y]
    return xi

#---------------ODE Dependent Stationary Ref Frame Animation Function---------------+

def plotV8(line_arr, txt, time, n, dist, phi, qa_2, qa_0, qa_n2):
    '''
    description:
    - the animation plotting function (not used for static outputs nor initializing the figure)
    - plotV8 called in every iteration/frame of the animation
    - updates the position of the perturbed NS using ODE solution curves (r, phi, qa)
    - use with findDynamicRad_V6
    --------------
    input: 
    line_arr (ndarray): array of 3 line objects with coordinates for: perturbed star, major axis, companion 
    txt (plt obj): plt.text() object for time stamp
    time (float): values from ode solution t array (unitless)
    n (int): frame index, used to moderate how fast the time stamp changes
    dist (float): value for given time from the ode solution curve r (orbital separation)
    phi (float): value for given time from the ode solution curve phi (separation angle)
    qa_2 (float): value for given time from the ode solution curve qa for m=2 
    qa_0 (float): value for given time from the ode solution curve qa for m=0
    qa_n2 (float): value for given time from the ode solution curve qa for m=-2
    '''

    t = time   # ODE unitless time for time stamp

    # calculate companion star center coordinates using ODE solution curves
    x_center = [dist*np.cos(phi)] 
    y_center = [dist*np.sin(phi)]

    # calculate xi coordinates in xy plane
    xi = findDynamicRad_V6(qa_2, qa_0, qa_n2)
    x = xi[0]
    y= xi[1]

    # find major axis start and end coordinates
    axPoints = findMajorAxis(x, y)
    x_ax = [-axPoints[0],axPoints[0]]
    y_ax =[-axPoints[1],axPoints[1]]
        
    # update timestamp
    if n%3 == 0:
        s = "t = " + str(t)
        txt.set_text(s)
        
    # update line objects for xi, major axes, and companion star
    update_x = [x, x_ax, x_center]
    update_y = [y, y_ax, y_center]
    for line, newx, newy in zip(line_arr, update_x, update_y):
        line.set_data(newx, newy)
            
#---------------ODE Dependent Stationary Ref Frame Plotting Functions---------------+

def plotV6(ax, dist, phi, qa_2, qa_0, qa_n2):
    '''
    description:
    - the static plotting function (not used for animating)
    - plotV6 called to initialize the figure that plotV8 will update
    - initial frame dependent on ODE solution curves
    - use with findDynamicRad_V6
    --------------
    input:
    ax (plt obj): axis of plot to be initialized
    dist (float): value for given time from the ode solution curve r (orbital separation)
    phi (float): value for given time from the ode solution curve phi (separation angle)
    qa_2 (float): value for given time from the ode solution curve qa for m=2 
    qa_0 (float): value for given time from the ode solution curve qa for m=0
    qa_n2 (float): value for given time from the ode solution curve qa for m=-2
    '''

    # calculate companion star center coordinates in first frame
    x_center = [dist*np.cos(phi)] 
    y_center = [dist*np.sin(phi)]
        
    # calculate xi coordinates in first frame
    xi = findDynamicRad_V6(qa_2, qa_0, qa_n2)
    x = xi[0]
    y = xi[1]
        
    # initial plot of xi and its major axis
    axPoints = findMajorAxis(x, y)
    line1, = ax.plot([-axPoints[0],axPoints[0]], [-axPoints[1],axPoints[1]], color='b')
    line0, = ax.plot(x, y, color='b')
        
    # initial plot of monopole & companion, no axes 
    ax.plot(x1, y1, color='r')
    line2, = ax.plot(x_center, y_center, color='black', marker='o')   

    return [line0, line1, line2]

def plotV10(ax, dist, phi, qa_2, qa_0, qa_n2):
    '''
    description:
    - plots only the quadrupole contribution and the companion NS 
    - use with findDynamicQuad
    - use similar to plotV6 (static frame, not for animation)
    --------------
    input:
    ax (plt obj): axis of plot to be initialized
    dist (float): value for given time from the ode solution curve r (orbital separation)
    phi (float): value for given time from the ode solution curve phi (separation angle)
    qa_2 (float): value for given time from the ode solution curve qa for m=2 
    qa_0 (float): value for given time from the ode solution curve qa for m=0
    qa_n2 (float): value for given time from the ode solution curve qa for m=-2
    '''

    # calculate companion star center coordinates in first frame
    x_center = [dist*np.cos(phi)] 
    y_center = [dist*np.sin(phi)]
        
    # calculate xi coordinates in first frame
    xi = findDynamicQuad(qa_2, qa_0, qa_n2)
    x = xi[0]
    y = xi[1]
        
    # plot of quadrupole components of xi and companion
    ax.plot(x, y, color='b')
    ax.plot(x_center, y_center, color='black', marker='o')   

    return ax

In [ ]:
#---------------Non-Rotating Ref Frame Static Output---------------+

n = 1326   # change time (0<n<1327) to isolate different animation frames
figure, axis = plt.subplots()
axis.set_xlim(-4,4)
axis.set_ylim(-4,4)
axis.set_aspect(1)
t_vals = sol.t
line1 = plotV6(axis, dist[n], sep_angle[n], qa_complex_2[n], qa_complex_0[n], qa_complex_n2[n])   # monopole + companion + deformed NS
line2 = plotV10(axis, dist[n], sep_angle[n], qa_complex_2[n], qa_complex_0[n], qa_complex_n2[n])   # quadrupole

In [10]:
#---------------Non-Rotating Ref Frame Animation---------------+
 
t0 = 950   # start time index (don't start at ODE start time because perturbation is so small)
figure, axis = plt.subplots()
axis.set_xlim(-4,4)
axis.set_ylim(-4,4)
axis.set_aspect(1) 

t_vals = []
t_len = len(sol.t) - t0
for i in range(t_len):
    t_vals.append(sol.t[t0+i])

lines = plotV6(axis, dist[t0], sep_angle[t0], qa_complex_2[t0], qa_complex_0[t0], qa_complex_n2[t0])
tf = t_vals[(len(t_vals)-1)]
s = "t = " + str(tf-t_vals[0])
txt = plt.text(-3, -3.75, s)

def update(frame):
    '''
    description:
    - updates the line objects for each frame of stationary frame animation
    - use with findDynamicRad_V6, plotV6, and plotV8
    --------------
    notes: 
    - adgust start time t0 as needed 
    - t0 < 200 may produce animation too large for Jupyter
    '''
    t = t_vals[frame]
    t_n = frame + t0
    dist_n = dist[t_n]
    sep_angle_n = sep_angle[t_n]
    qa_2 = qa_complex_2[t_n]
    qa_0 = qa_complex_0[t_n]
    qa_n2 = qa_complex_n2[t_n]
    plotV8(lines, txt, tf-t, t_n, dist_n, sep_angle_n, qa_2, qa_0, qa_n2)

ani = FuncAnimation(figure, update, frames=len(t_vals), interval=150, blit=False, repeat=False) 
# HTML(ani.to_jshtml())   # display in notebook
ani.save('C:/Users/smoge/binary_NS_model/BNS_stationary_frame_pm.gif', writer='pillow', fps=20)   # save externally
plt.close()

## Corotating Reference Frame Animations
In these animations, the observer rotates with the deformed star. We apply a rotation matrix to the coodinates defined in the stationary ref frame. The transformed coordinates $(x',y')$ are calculated as follows:
$$x' = x\cos(\phi_{ref}) + y\sin(\phi_{ref})$$
$$y' = x\sin(\phi_{ref}) + y\cos(\phi_{ref})$$

where $\phi_{ref}$ is the tidal lag angle of the companion.

In [11]:
#---------------Calculation Functions---------------+

def xy_transform(x, y, ref):
    '''
    description:
    - applies a standard rotation matrix to the xy coordinates using the lag angle of the companion
    --------------
    input:
    x (ndarray): 1D array of x coordinates
    y (ndarray): 1D array of y coordinates
    ref (float): value of lag angle of companion at a time given in the update function
    '''
    x_prime = x*np.cos(ref) + y*np.sin(ref)
    y_prime = -x*np.sin(ref) + y*np.cos(ref)
    return [x_prime, y_prime]

#---------------ODE Dependent Corotating Ref Frame Animation Functions---------------+

def plotV11(line_arr, txt, time, n, dist, phi, qa_2, qa_0, qa_n2, pm=True):
    '''
    description:
    - the animation plotting function for a corotating reference frame (the animation looks like it doesn't spin)
    - plotV11 called in every iteration/frame of the animation
    - updates the position of the perturbed NS using ODE solution curves (r, phi, qa)
    - use with findDynamicRad_V6 and xy_transform
    - similar to plotV8, but with a coordinate transform
    --------------
    input: 
    line_arr (ndarray): array of 4 line objects with coordinates 
        - if pm=False: perturbed star, major axis, perturbed companion, major axis
        - if pm=True: perturbed star, major axis, companion center
    txt (plt obj): plt.text() object for time stamp
    time (float): values from ode solution t array (unitless)
    n (int): frame index, used to moderate how fast the time stamp changes
    dist (float): value for given time from the ode solution curve r (orbital separation)
    phi (float): value for given time from the ode solution curve phi (separation angle)
    qa_2 (float): value for given time from the ode solution curve qa for m=2 
    qa_0 (float): value for given time from the ode solution curve qa for m=0
    qa_n2 (float): value for given time from the ode solution curve qa for m=-2
    pm (bool): indicates whether the companion is plotted as a point mass, default: pm=True
    '''

    t = time   # ODE unitless time

    # calculate transformed point mass companion center coordinates
    x_c = dist*np.cos(phi)
    y_c = dist*np.sin(phi)
    xy_ct = xy_transform(x_c, y_c, phi)   # transformed coordinates
    x_ct = [xy_ct[0]]
    y_ct = [xy_ct[1]]

    # calculate transformed xi coordinates in xy plane
    xi = findDynamicRad_V6(qa_2, qa_0, qa_n2)
    x = xi[0]
    y= xi[1] 
    xy_t = xy_transform(x, y, phi)   # transformed coordinates
    x_t = xy_t[0]
    y_t = xy_t[1]

    # find major axis start and end coordinates
    axPoints = findMajorAxis(x_t, y_t)
    x_ax = [-axPoints[0],axPoints[0]]
    y_ax =[-axPoints[1],axPoints[1]]
        
    # update timestamp
    if n%3 == 0:
        s = "t = " + str(t)
        txt.set_text(s)

    if pm==False:   # update perturbed companion coordinates for non point mass
        # perturbed companion coordinates
        x_ct = x_t + x_ct*np.ones(len(x_t))
        y_ct = y_t + y_ct*np.ones(len(y_t))

        # major axis of perturbed companion
        ax_xc = [[xy_ct[0]]-axPoints[0], axPoints[0]+[xy_ct[0]]]
        ax_yc = [[xy_ct[1]]-axPoints[1], axPoints[1]+[xy_ct[1]]]

        # update line objects for xi, major axis, perturbed companion, companion axis
        update_x = [x_t, x_ax, x_ct, ax_xc]
        update_y = [y_t, y_ax, y_ct, ax_yc]
        for line, newx, newy in zip(line_arr, update_x, update_y):
            line.set_data(newx, newy)
        
    else:   # update line objects for xi, major axis, and point mass companion star
        update_x = [x_t, x_ax, x_ct]
        update_y = [y_t, y_ax, y_ct]
        for line, newx, newy in zip(line_arr, update_x, update_y):
            line.set_data(newx, newy)
        
#---------------ODE Dependent Corotating Ref Frame Plotting Functions---------------+

def plotV12(ax, dist, phi, qa_2, qa_0, qa_n2, pm=True):
    '''
    description:
    - the static plotting function for a corotating reference frame
    - plotV12 called to initialize the figure that plotV11 will update
    - initial frame dependent on ODE solution curves
    - use with findDynamicRad_V6 and xy_transform
    --------------
    input:
    ax (plt obj): axis of plot to be initialized
    dist (float): value for given time from the ode solution curve r (orbital separation)
    phi (float): value for given time from the ode solution curve phi (separation angle)
    qa_2 (float): value for given time from the ode solution curve qa for m=2 
    qa_0 (float): value for given time from the ode solution curve qa for m=0
    qa_n2 (float): value for given time from the ode solution curve qa for m=-2
    pm (bool): indicates whether the companion is plotted as a point mass, default: pm=True
    '''
        
    # calculate transformed xi coordinates in first frame
    xi = findDynamicRad_V6(qa_2, qa_0, qa_n2)
    x = xi[0]
    y = xi[1]
    xy_t = xy_transform(x, y, phi)   # transformed coordinates
    x_t = xy_t[0]
    y_t = xy_t[1]
        
    # initial plot of xi and its major axis
    axPoints = findMajorAxis(x_t, y_t)
    line1, = ax.plot([-axPoints[0],axPoints[0]], [-axPoints[1],axPoints[1]], color='b')
    line0, = ax.plot(x_t, y_t, color='b')

    # calculate transformed point mass companion center coordinates in first frame
    x_c = dist*np.cos(phi) 
    y_c = dist*np.sin(phi)
    xy_ct = xy_transform(x_c, y_c, phi)   # coordinate transform

    # monopole
    ax.plot(x1, y1, color='r')

    if pm==False:   # perturbed companion
        # initial perturbed companion coordinates
        x_ct = x_t + xy_ct[0]*np.ones(len(x_t))
        y_ct = y_t + xy_ct[1]*np.ones(len(y_t))
        line2, = ax.plot(x_ct, y_ct, color='black')

        # initial major axis of perturbed companion
        ax_xc = [xy_ct[0]-axPoints[0], axPoints[0] + xy_ct[0]]
        ax_yc = [xy_ct[1]-axPoints[1], axPoints[1] + xy_ct[1]]
        line3, = ax.plot(ax_xc, ax_yc, color='black')
        ax.plot([-2, 5], [0, 0], color='black', alpha=0.5)
        
        return [line0, line1, line2, line3]   # perturbed star, major axis, perturbed companion, major axis
        
    else:   # point mass companion, pm=True   
        # initial plot companion, no axes 
        ax.plot([-2, 4.5], [0, 0], color='black', alpha=0.5)
        line2, = ax.plot(xy_ct[0], xy_ct[1], color='black', marker='o') 

        return [line0, line1, line2]   # perturbed star, major axis, point mass companion

In [12]:
#---------------Corotating Ref Frame Animation, Point Mass Companion---------------+

# need to run all calc functions from stationary ref frame section to run this animation
t0 = 950   # adjust start time 
figure, axis = plt.subplots()
axis.set_xlim(-2,4.5)
axis.set_ylim(-2,2)
axis.set_aspect(1) 

t_vals = []
t_len = len(sol.t) - t0
for i in range(t_len):
    t_vals.append(sol.t[t0+i])

lines = plotV12(axis, dist[t0], sep_angle[t0], qa_complex_2[t0], qa_complex_0[t0], qa_complex_n2[t0])
tf = t_vals[(len(t_vals)-1)]
s = "t = " + str(tf-t_vals[0])
txt = plt.text(-1.5, -1.75, s)

def update(frame):
    '''
    description:
    - updates the line objects for each frame of corotating animation
    - POINT MASS COMPANION version (pm=True)
    - use with findDynamicRad_V6, plotV11, and plotV12
    --------------
    notes: 
    - POINT MASS COMPANION
    - adgust start time t0 as needed 
    - t0 < 200 may produce animation too large for Jupyter
    '''
    t = t_vals[frame]
    t_n = frame + t0
    dist_n = dist[t_n]
    sep_angle_n = sep_angle[t_n]
    qa_2 = qa_complex_2[t_n]
    qa_0 = qa_complex_0[t_n]
    qa_n2 = qa_complex_n2[t_n]
    plotV11(lines, txt, tf-t, t_n, dist_n, sep_angle_n, qa_2, qa_0, qa_n2)

ani = FuncAnimation(figure, update, frames=len(t_vals), interval=150, blit=False, repeat=False) 
# HTML(ani.to_jshtml())   # display in notebook
ani.save('C:/Users/smoge/binary_NS_model/BNS_corot_frame_pm.gif', writer='pillow', fps=20)   # save externally
plt.close()

In [13]:
#---------------Corotating Ref Frame Animation, Perturbed Companion---------------+

t0 = 950   # adjust start time
figure, axis = plt.subplots()
axis.set_xlim(-2,5)
axis.set_ylim(-2,2)
axis.set_aspect(1) 

t_vals = []
t_len = len(sol.t) - t0
for i in range(t_len):
    t_vals.append(sol.t[t0+i])

lines = plotV12(axis, dist[t0], sep_angle[t0], qa_complex_2[t0], qa_complex_0[t0], qa_complex_n2[t0], pm=False)
tf = t_vals[(len(t_vals)-1)]
s = "t = " + str(tf-t_vals[0])
txt = plt.text(-1.5, -1.75, s)

def update(frame):
    '''
    description:
    - updates the line objects for each frame of corotating animation
    - TIDALLY PERTURBED COMPANION version (pm=False)
    - use with findDynamicRad_V6, plotV11, and plotV12
    --------------
    notes: 
    - adgust start time t0 as needed 
    - t0 < 200 may produce animation too large for Jupyter
    '''
    t = t_vals[frame]
    t_n = frame + t0
    dist_n = dist[t_n]
    sep_angle_n = sep_angle[t_n]
    qa_2 = qa_complex_2[t_n]
    qa_0 = qa_complex_0[t_n]
    qa_n2 = qa_complex_n2[t_n]
    plotV11(lines, txt, tf-t, t_n, dist_n, sep_angle_n, qa_2, qa_0, qa_n2, pm=False)

ani = FuncAnimation(figure, update, frames=len(t_vals), interval=150, blit=False, repeat=False) 
# HTML(ani.to_jshtml())   # display in notebook
ani.save('C:/Users/smoge/binary_NS_model/BNS_corot_frame.gif', writer='pillow', fps=20)   # save externally
plt.close()